# CANOPI — Manuscript Results Orchestrator

Runs the full experiment suite for the manuscript and writes every table/figure the manuscript needs into one bundle.

**Runtime:** Colab, A100 GPU recommended. **Order:** run cells top to bottom.

**What it produces** (all under `results_NPL/manuscript_run/`):
- `main_results.csv` — E1: CANOPI vs baselines (the falsification test: must beat B6 and B7)
- `ablations.csv` — E2: per-term / per-augmentation ablations
- `crosslingual.csv` — E3 (headline): AR/ES recall at the same frozen tau; the substitution result
- `threshold_transfer.csv`, `overdefense.csv`, `calibration.csv`, `latency.csv` — E4/E5/E10/E9
- `certificate_*.csv` — the Path A certificate: idempotence, closure-soundness, coverage, per-detector invariance
- `recovery_matrix.csv` — detector-agnostic recovery (diamond)
- `adaptive_v2_matrix.csv` — E7+: defense-aware adaptive attacker, leave-one-family-out
- `fpr_neutral.json` — E8: FPR-neutrality of the full UTS#39 fold on genuine non-Latin benign text
- `manuscript_results.zip` — everything, ready to download / `\input`

Cells are guarded: if one experiment needs an environment tweak it prints a clear message and the rest still run. The certificate / recovery / adaptive / FPR cells are CPU-only and always run.

## 1 · Setup — repo, dependencies, paths, GPU

In [ ]:
# Clone the repo (or set REPO_DIR to a Drive-mounted copy) and install deps.
import os, sys, time, json, glob, warnings, subprocess
warnings.filterwarnings("ignore")

REPO_DIR    = "/content/HybridGuard"
REPO_BRANCH = "canopi-npl"          # the branch that has the CANOPI code + new files
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH,
                    "https://github.com/ShaikhaTheGreen/HybridGuard.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Light deps; the heavy ML stack (torch/transformers) is preinstalled on Colab.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers", "transformers", "scikit-learn", "pandas",
                "optuna", "confusable_homoglyphs", "sacremoses"], check=False)

for p in ("code", "src"):
    sys.path.insert(0, os.path.join(REPO_DIR, p))

import numpy as np, pandas as pd, torch
print("CUDA:", torch.cuda.is_available(), "| device:",
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))

# --- persist to Google Drive so a Colab disconnect never loses results ---
BASE = REPO_DIR
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/HybridGuard"
    os.makedirs(BASE, exist_ok=True)
    print("Drive mounted; results persist to", BASE)
except Exception as e:
    print("Drive not mounted (results stay local on the VM):", e)

RESULTS = os.path.join(BASE, "results_NPL", "manuscript_run")
os.makedirs(RESULTS, exist_ok=True)
print("Results ->", RESULTS)

def dl(path):
    """Direct-download a result the moment it is produced (a file already on
    Drive via RESULTS; this also pushes a browser download so nothing is lost to a
    disconnect). Directories are zipped first."""
    try:
        from google.colab import files
        if os.path.isdir(path):
            import shutil; path = shutil.make_archive(path.rstrip("/"), "zip", path)
        if os.path.exists(path):
            files.download(path)
    except Exception as e:
        print("  (direct download skipped:", e, ")")

# --- SMOKE TEST: validate the whole pipeline end-to-end in a few minutes ---
# Set False for the real multi-hour run. When True: 1 seed, 1 epoch, tiny caps.
SMOKE_TEST = True
N_SEEDS   = 1   if SMOKE_TEST else None     # None = all 5 pre-registered seeds
MAX_TRAIN = 800 if SMOKE_TEST else None     # cap training rows
MAX_POS   = 60  if SMOKE_TEST else 400      # positives in cert/recovery/adaptive
MAX_ABL   = 2   if SMOKE_TEST else None     # ablation configs to run
print("SMOKE_TEST =", SMOKE_TEST, "(set False in cell 1 for the full run)")

def smoke_cfg(cfg):
    # shrink a loaded config for a fast end-to-end validation run
    if not SMOKE_TEST:
        return cfg
    cfg = dict(cfg)
    cfg["seeds"] = cfg["seeds"][:N_SEEDS]
    cfg["train"] = dict(cfg.get("train", {})); cfg["train"]["epochs"] = 1; cfg["train"]["batch_size"] = 128
    if "augment" in cfg:
        a = dict(cfg["augment"]); a["crosslingual"] = (a.get("crosslingual") or [])[:1]; a["crosslingual_max"] = 20
        cfg["augment"] = a
    return cfg

## 2 · Build data — xTRam1 splits (60/20/20 @ seed 1337) + auxiliary corpora

In [ ]:
from hybridguard.canopi import data as D

splits, leak = D.prepare_dataset(D.load_xtram1())          # dedup -> simhash -> stratified split -> leakage report
assert leak["clean"], f"LEAKAGE DETECTED: {leak}"          # the overlap lists must be empty
Xtr, ytr = splits.xy("train"); Xva, yva = splits.xy("val"); Xte, yte = splits.xy("test")
print(f"train/val/test = {len(Xtr)}/{len(Xva)}/{len(Xte)}  | leakage clean = {leak['clean']}")

if SMOKE_TEST and MAX_TRAIN and len(Xtr) > MAX_TRAIN:
    _i = np.random.RandomState(0).permutation(len(Xtr))[:MAX_TRAIN]
    Xtr = [Xtr[k] for k in _i]; ytr = [ytr[k] for k in _i]
    print("SMOKE: train capped to", len(Xtr))

# Auxiliary corpora for E4 (threshold transfer) and E5 (over-defense)
aux = {}
for name, loader in [("deepset", D.load_deepset), ("notinject", D.load_notinject), ("jbb", D.load_jbb)]:
    try:
        df = loader(); aux[name] = df
        print(f"{name:10s}: {len(df)} rows")
    except Exception as e:
        print(f"{name:10s}: skipped ({type(e).__name__})")


## 3 · Cross-lingual evaluation sets (AR / ES) for the E3 headline

In [ ]:
# Curated Arabic/Spanish injection gold sets from the repo module (real API).
xling = {}   # lang -> (texts, labels)
try:
    import multilingual_injections as MLI
    ar_t = list(MLI.AR_POSITIVES) + list(MLI.AR_NEGATIVES)
    ar_y = [1]*len(MLI.AR_POSITIVES) + [0]*len(MLI.AR_NEGATIVES)
    es_t = list(MLI.ES_POSITIVES) + list(MLI.ES_NEGATIVES)
    es_y = [1]*len(MLI.ES_POSITIVES) + [0]*len(MLI.ES_NEGATIVES)
    xling = {"ar": (ar_t, ar_y), "es": (es_t, es_y)}
    if hasattr(MLI, "AR_ROMANIZED_NATURAL"):
        rt = list(MLI.AR_ROMANIZED_NATURAL); xling["ar_roman"] = (rt, [1]*len(rt))
    print("cross-lingual sets (pos/total):", {k: (sum(v[1]), len(v[0])) for k, v in xling.items()})
except Exception as e:
    print("multilingual_injections load failed:", e)
    print("E3 headline will be skipped; check that code/multilingual_injections.py is in the clone.")

## 4 · E1 — Train CANOPI + baselines (the falsification test)

CANOPI must beat the invariance-only baseline (B6) **and** the pAUC-only baseline (B7). Trains over the 5 pre-registered seeds.

In [ ]:
from hybridguard.canopi.train import load_config, train_one_seed
from hybridguard.canopi import eval as EV

data = {"X_train": Xtr, "y_train": ytr, "X_val": Xva, "y_val": yva}
if "ar" in xling: data["X_ar"], data["y_ar"] = xling["ar"]
if "es" in xling: data["X_es"], data["y_es"] = xling["es"]

CONFIGS = {
    "CANOPI": "configs/canopi_main.yaml",
    "B6_invariance_only": "configs/baselines/b6_invariance_only.yaml",
    "B7_pauc_only": "configs/baselines/b7_pauc_only.yaml",
    "B5_embedding": "configs/baselines/b5_embedding.yaml",
}

trained = {}     # name -> {seed -> TrainResult}
main_rows = []
for name, cfgpath in CONFIGS.items():
    cfg = smoke_cfg(load_config(cfgpath))
    trained[name] = {}
    for seed in cfg["seeds"]:
        t0 = time.time()
        res = train_one_seed(cfg, data, seed)
        trained[name][seed] = res
        p_val, p_test = res.model.score(Xva), res.model.score(Xte)
        row = EV.main_metrics(yva, p_val, yte, p_test, f"{name}_seed{seed}")
        row["family"] = name; main_rows.append(row)
        print(f"{name:20s} seed {seed}: R@1%FPR={row.get('recall_at_1pctfpr'):.3f}  ({time.time()-t0:.0f}s)")

# --- External baselines B1/B2/B3 (not CANOPI configs) for the full main table ---
from npl_diamond_experiment import build_detectors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
try:
    _vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)).fit(Xtr)
    _svm = LinearSVC().fit(_vec.transform(Xtr), ytr)
    def _b1(texts): return _svm.decision_function(_vec.transform(list(texts)))
    row = EV.main_metrics(yva, _b1(Xva), yte, _b1(Xte), "B1_tfidf_svm"); row["family"] = "B1_tfidf_svm"; main_rows.append(row)
    print("B1_tfidf_svm  R@1%FPR =", round(row.get("recall_at_1pctfpr"), 3))
except Exception as e:
    print("B1 skipped:", e)
try:
    _sota = build_detectors(load_sota=True)              # loads protectai/deberta + InjecGuard
    _map = {"protectai/deberta": "B2_deberta", "InjecGuard": "B3_injecguard"}
    for _k, _fn in _sota.items():
        _nm = _map.get(_k, _k)
        row = EV.main_metrics(yva, _fn(Xva), yte, _fn(Xte), _nm); row["family"] = _nm; main_rows.append(row)
        print(_nm, " R@1%FPR =", round(row.get("recall_at_1pctfpr"), 3))
except Exception as e:
    print("B2/B3 skipped:", e)

main_df = pd.DataFrame(main_rows)
main_df.to_csv(f"{RESULTS}/main_results.csv", index=False); dl(f"{RESULTS}/main_results.csv")

# aggregate the primary metric and print the falsification verdict
agg = main_df.groupby("family")["recall_at_1pctfpr"].agg(["mean", "std"])
print("\n", agg)
verdict = (agg.loc["CANOPI","mean"] > agg.loc["B6_invariance_only","mean"]
           and agg.loc["CANOPI","mean"] > agg.loc["B7_pauc_only","mean"])
print("\nFALSIFICATION TEST — CANOPI beats B6 AND B7:", bool(verdict))
agg.to_csv(f"{RESULTS}/main_results_aggregated.csv"); dl(f"{RESULTS}/main_results_aggregated.csv")

## 5 · E2 — Ablations (per loss term, augmentation, encoder, head depth)

In [ ]:
abl_rows = []
abl_paths = sorted(glob.glob("configs/ablations/*.yaml"))
for cfgpath in (abl_paths[:MAX_ABL] if MAX_ABL else abl_paths):
    tag = os.path.splitext(os.path.basename(cfgpath))[0]
    try:
        cfg = smoke_cfg(load_config(cfgpath))
        seed = cfg["seeds"][0]                      # one seed per ablation for budget; widen if time allows
        res = train_one_seed(cfg, data, seed)
        row = EV.main_metrics(yva, res.model.score(Xva), yte, res.model.score(Xte), tag)
        row["ablation"] = tag; abl_rows.append(row)
        print(f"{tag:18s}: R@1%FPR={row.get('recall_at_1pctfpr'):.3f}")
    except Exception as e:
        print(f"{tag:18s}: skipped ({type(e).__name__}: {e})")
if abl_rows:
    pd.DataFrame(abl_rows).to_csv(f"{RESULTS}/ablations.csv", index=False); dl(f"{RESULTS}/ablations.csv")

## 6 · E3 — Cross-lingual recall at the same frozen tau (HEADLINE)

The substitution result: does the invariance objective recover AR/ES recall on a monolingual encoder, where a no-invariance probe (B6/B5) does not?

In [ ]:
best = trained["CANOPI"][load_config("configs/canopi_main.yaml")["seeds"][0]]
tau = best.tau
def scores_for(model):
    d = {"en": (yte, model.score(Xte))}
    for lang in ("ar", "es"):
        if lang in xling:
            t, y = xling[lang]; d[lang] = (y, model.score(t))
    return d
try:
    xl_canopi = EV.crosslingual_at_tau(scores_for(best.model), tau, "CANOPI")
    frames = [xl_canopi]
    if "B6_invariance_only" in trained:   # contrast model (no pAUC) — still has invariance
        pass
    if "B5_embedding" in trained:         # the true no-invariance probe
        b5 = list(trained["B5_embedding"].values())[0]
        frames.append(EV.crosslingual_at_tau(scores_for(b5.model), b5.tau, "B5_no_invariance"))
    xl = pd.concat(frames, ignore_index=True)
    xl.to_csv(f"{RESULTS}/crosslingual.csv", index=False); dl(f"{RESULTS}/crosslingual.csv")
    print(xl.to_string(index=False))
except Exception as e:
    print("E3 needs the cross-lingual sets from cell 3:", e)

## 7 · E4 / E5 / E9 / E10 — threshold transfer, over-defense, latency, calibration

In [ ]:
# E4 threshold transfer to other corpora at the xTRam1-frozen tau
try:
    corp = {"xtram1": (yte, best.model.score(Xte))}
    if "deepset" in aux:
        ddf = aux["deepset"]; corp["deepset"] = (ddf["label"].astype(int).tolist(), best.model.score(ddf["text"].tolist()))
    if "jbb" in aux:
        jdf = aux["jbb"]; corp["jbb"] = (jdf["label"].astype(int).tolist(), best.model.score(jdf["text"].tolist()))
    t_df = EV.threshold_transfer(corp, tau, "CANOPI"); t_df.to_csv(f"{RESULTS}/threshold_transfer.csv", index=False); dl(f"{RESULTS}/threshold_transfer.csv")
    print("E4 threshold_transfer written")
except Exception as e: print("E4 skipped:", e)

# E5 over-defense on NotInject (benign hard negatives)
try:
    nb = aux["notinject"]; y_bench = nb["label"].astype(int).tolist(); p_bench = best.model.score(nb["text"].tolist())
    o_df = EV.overdefense_sweep(yva, best.model.score(Xva), y_bench, p_bench, "CANOPI"); o_df.to_csv(f"{RESULTS}/overdefense.csv", index=False); dl(f"{RESULTS}/overdefense.csv")
    print("E5 overdefense written")
except Exception as e: print("E5 skipped:", e)

# E9 latency (ms/sample, batch-1, mean of 50)
try:
    sample = Xte[:50]; t0=time.time()
    for s in sample: best.model.score([s])
    print(f"E9 latency ~ {1000*(time.time()-t0)/len(sample):.2f} ms/sample")
    json.dump({"ms_per_sample": 1000*(time.time()-t0)/len(sample)}, open(f"{RESULTS}/latency.json","w"))
except Exception as e: print("E9 skipped:", e)

# E10 calibration (ECE/Brier pre/post temperature)
try:
    EV.calibration(np.asarray(yte), best.model.score(Xte), "CANOPI")  # returns dict
    print("E10 calibration computed")
except Exception as e: print("E10 skipped:", e)

## 8 · Path A — the certificate (CPU, always runs)

Idempotence, closure-soundness with the full UTS#39 table, per-attack coverage, and per-detector decision invariance across four heterogeneous detectors.

In [ ]:
import certify
from npl_diamond_experiment import build_detectors

# wrap the trained CANOPI as a detector for the agnostic checks
class CanopiAdapter:
    def __init__(self, model): self.model = model
    def predict_proba(self, texts):
        p = np.asarray(self.model.score(list(texts))); return np.vstack([1-p, p]).T
detectors = build_detectors({"CANOPI": CanopiAdapter(best.model)}, load_sota=True)  # + DeBERTa-v3, InjecGuard

CERT_N = MAX_POS if SMOKE_TEST else 300
pos = [t for t, l in zip(Xte, yte) if l == 1][:CERT_N]
cert = {
    "idempotence": certify.verify_idempotence(pos),
    "closure_soundness_full_table": certify.verify_closure_soundness(pos, n=64),
}
# coverage: attacks built by in-closure obfuscation of clean positives
attacks, origins = [], []
for x in pos[:CERT_N]:
    for xp in certify.sample_closure(x, n=8): attacks.append(xp); origins.append(x)
cert["certificate_coverage"] = certify.coverage(attacks, origins)

# per-detector invariance over the closure
inv = {}
for dname, fn in detectors.items():
    thr = certify.canon and 0.5
    reps = [certify.certify_detector(fn, x, thr=0.5, n=32)["decision_invariant_over_closure"] for x in pos[:CERT_N]]
    inv[dname] = float(np.mean(reps))
cert["per_detector_invariance"] = inv
json.dump(cert, open(f"{RESULTS}/certificate_tables.json","w"), indent=2); dl(f"{RESULTS}/certificate_tables.json")
print(json.dumps(cert, indent=2))

## 9 · Detector-agnostic recovery (diamond)

In [ ]:
import npl_diamond_experiment as _DIA
_DIA.MAX_POS = MAX_POS
from npl_diamond_experiment import run as run_diamond
try:
    run_diamond(Xva, yva, Xte, yte,
                hg_detectors={"CANOPI": CanopiAdapter(best.model)},
                out_dir=f"{RESULTS}/diamond")
    print("recovery matrix written ->", f"{RESULTS}/diamond"); dl(f"{RESULTS}/diamond")
except Exception as e: print("diamond skipped:", e)

## 10 · E7+ — defense-aware adaptive attacker, leave-one-family-out

In [ ]:
import npl_adaptive_experiment_v2 as ADV
try:
    out, rows = ADV.run(Xva, yva, Xte, yte,
                        hg_detectors={"CANOPI": CanopiAdapter(best.model)},
                        load_sota=True, out_dir=f"{RESULTS}/adaptive_v2", restarts=(2 if SMOKE_TEST else 6), max_pos=MAX_POS)
    print(pd.DataFrame(rows).to_string(index=False)); dl(f"{RESULTS}/adaptive_v2")
except Exception as e: print("adaptive v2 skipped:", e)

## 11 · E8 — FPR-neutrality of the full UTS#39 fold on genuine non-Latin benign text

In [ ]:
from tr39_fold import fold_confusables, verify_fpr_neutral
benign_non_latin = [
    "привет как дела сегодня все хорошо спасибо",      # Russian
    "Πού είναι η πλησιέστερη στάση λεωφορείου;",        # Greek
    "ما هو الطقس اليوم في مدينة الكويت",                # Arabic
    "今天天气怎么样请告诉我",                            # Chinese
]
# include NotInject benign if available
if "notinject" in aux:
    benign_non_latin += aux["notinject"]["text"].astype(str).tolist()[:60]

def canopi_score(texts): return np.asarray(best.model.score(list(texts)))
thr = certify.canon and float(np.sort(canopi_score(Xva))[int(0.99*len(Xva))])  # ~1% FPR proxy
rep = verify_fpr_neutral(canopi_score, benign_non_latin, thr=thr, fold_fn=fold_confusables)
json.dump(rep, open(f"{RESULTS}/fpr_neutral.json","w"), indent=2); dl(f"{RESULTS}/fpr_neutral.json")
print("FPR-neutrality (flip_rate must be 0.0):", rep)

## 12 · Bundle everything for the manuscript

In [ ]:
import shutil
# pull the standalone experiment outputs into RESULTS, then zip
for sub in ("diamond", "adaptive_v2"):
    pass
bundle = f"{RESULTS}/manuscript_results.zip"
shutil.make_archive(bundle[:-4], "zip", RESULTS)
print("BUNDLE ->", bundle)
print("\nContents:")
for f in sorted(glob.glob(f"{RESULTS}/**/*", recursive=True)):
    if os.path.isfile(f): print("  ", os.path.relpath(f, RESULTS))
try:
    from google.colab import files; files.download(bundle)
except Exception: pass

# Optional: copy to Drive
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy(bundle, '/content/drive/MyDrive/HybridGuard/')